# JOGO DO DILEMA DOS PRISIONEIROS

## PESQUISADOR ORIENTADOR E ALUNO

**Aluno:** Gustavo Junqueira Colas  
**Professores:** Daniel Roberto Cassar, James Moraes de Almeida, Leandro Nascimento Lemos  
**Disciplina:** Práticas em Ciência de Dados (PCD)  
**Instituição:** ILUM - Escola de Ciência / CNPEM  
**Data:** Junho de 2026

## RESUMO

O presente projeto teve como objetivo desenvolver um jogo interativo do Dilema do Prisioneiro utilizando a linguagem Python e a biblioteca Pygame. Além de apresentar os conceitos fundamentais da Teoria dos Jogos, o programa leva o usuário a interagir com diferentes estratégias computacionais, observando como comportamentos distintos influenciam os resultados obtidos ao longo das partidas. O projeto também buscou explorar conceitos importantes da programação, como estruturas de dados, funções, animação por sprites e construção de interfaces gráficas.

## INTRODUÇÃO

### O DILEMA DOS PRISIONEIROS

O Dilema dos Prisioneiros é um dos problemas mais famosos da Teoria dos Jogos. O cenário clássico consiste em dois suspeitos que são presos e interrogados separadamente. Cada um pode escolher entre duas ações - **COOPERAR** (não delatar seu cúmplice) ou **TRAIR** (delatar o cúmplice). As penas resultantes dependem das escolhas combinadas:

|PRISIONEIRO 1 \| PRISIONEIRO 2 | COOPERAR | TRAIR |
|---|---|---|
| **COOPERAR** | 2 anos \| 2 anos | 10 anos \| 1 ano |
| **TRAIR** | 1 ano \| 10 anos | 5 anos \| 5 anos |

O paradoxo central do problema está no seguinte: se ambos cooperassem, a pena total seria minimizada (4 anos somados). Porém, do ponto de vista individual e racional, ou seja, que supõe que o outro jogador também seja racional, trair é sempre a decisão ótima: independentemente do que o outro faça, trair resulta em uma pena menor para quem trai. Esse equilíbrio, em que ambos traem e ambos saem prejudicados, é conhecido como Equilíbrio de Nash: nenhum dos jogadores tem incentivo individual para mudar sua estratégia unilateralmente.

No entanto, o jogo torna-se realmente interessante quando é jogado repetidamente, problema conhecido como **Dilema dos Prisioneiros Iterado**. A possibilidade de aprender com o histórico das rodadas anteriores abre espaço para estratégias mais sofisticadas, como a cooperação condicionada ao comportamento do adversário. Nesse contexto, estratégias como o "Imitador" (ou "Tit-for-Tat", em inglês) - que começa cooperando e depois imita a última jogada do oponente - se mostram surpreendentemente eficazes.

### MOTIVAÇÃO

O primeiro fator que motivou este projeto foi o interesse pessoal em Teoria dos Jogos, que despertou por meio de vídeos e leituras acerca da tomada de decisões. O Dilema dos Prisioneiros, em particular, foi escolhido como ponto focal do jogo por ser ao mesmo tempo simples de enunciar e profundamente contraintuitivo: para casos individuais, a escolha "racional" leva ambos os jogadores a um resultado pior do que o que obteriam se cooperassem. Isso tem implicações em áreas que se estendem da economia à política, da biologia evolutiva às relações internacionais. A análise por meio da programação do dilema dos prisioneiros serviu para iluminar contextos reais, como "por que vale a pena para um organismo intrinsecamente egoísta cooperar?", "como dois países inimigos podem parar com uma escalada armamentista?", além de muitas outras questões aplicadas.

O segundo fator que me impulsionou foi a busca por um projeto que me desafiasse tecnicamente. Sendo um aluno que teve seu primeiro contato com programação no início desse ano, criar uma interface gráfica interativa com animações, múltiplos estados de tela e lógica de bots representou um projeto bastante instigante. O jogo acabou sendo uma forma de implementar na prática conceitos como funções, variáveis globais, estruturas condicionais encadeadas e o uso de bibliotecas externas, das quais vale dar um enfoque à biblioteca "pygame" (cujo uso será mais bem esclarecido posteriormente).

## ESTRUTURA DO PROGRAMA

### INTERFACE GRÁFICA - FUNCIONAMENTO PRINCIPAL

A interface foi construída inteiramente com a biblioteca ***Pygame*** (em conjunto com outras duas bibliotecas usadas pontualemnte, cujo uso está explicitado por comentários no código), que permite criar janelas gráficas, capturar eventos do usuário (cliques e teclas) e redesenhar a tela a cada quadro. A janela do jogo tem dimensões de 700 x 600 *pixels* e é atualizada continuamente dentro de um loop principal. Foram definidas três fontes de texto em tamanhos diferentes - `arial`, `arial_pequeno` e `arial_huge` - para compor os diferentes elementos visuais das telas.

A navegação entre as telas do jogo é controlada por setas do teclado (`>` para avançar, `<` para voltar) e pela tecla de espaço para sair. A cada nova tela, funções específicas de desenho são chamadas para renderizar os elementos corretos.

O menu inicial apresenta uma animação de entrada: uma sequência de 5 sprites de uma cela sendo exibidos quadro a quadro, criando a ilusão de que ela está abrindo. Os sprites são armazenados em uma lista e percorridos com base em um contador de tempo (`tempo_frame`), que aumenta a cada ciclo do loop e avança para o próximo frame quando atinge um limite definido por `velocidade_animacao`.

Após o menu, o jogo exibe um jornal fictício - uma imagem que contextualiza a narrativa do Dilema dos Prisioneiros dentro de um assalto a banco -, o que serve como uma forma de deixar o jogador imerso no jogo. O jornal é seguido por uma tela de instruções com a tabela de payoffs e, em seguida, a tela de seleção de fases. Durante as partidas, a tela de jogo exibe dois botões clicáveis que representam a escolha do jogador (**COOPERAR** e **TRAIR**), um painel central que exibe as escolhas do jogador e do bot, o resultado da rodada em "anos de cadeia" e um placar que acumula a pena total do jogador. Os botões respondem visualmente ao cursor: ficam verdes ao passar o mouse sobre COOPERAR, e vermelhos ao passar sobre TRAIR.

O bloco de código abaixo inicializa o Pygame, define as dimensões da janela, as fontes de texto, as cores e os retângulos que compõem os botões e os principais elementos da interface. Os retângulos (`pg.Rect`) não são apenas figuras visuais: eles também funcionam como áreas de detecção de clique, por meio do método `.collidepoint(mouse_pos)`, que verifica se a posição do cursor está dentro do retângulo. Os elementos estão descritos por comentários.

In [1]:
import pygame as pg # usei para montar a interface gráfica
from random import choice # usei para determinar o comportamento de um dos bots
import sys # biblioteca utilizada somente para evitar algum erro ao fechar o programa


pg.init() # inicializa o Pygame

largura = 700
altura = 600

tela = pg.display.set_mode((largura, altura)) # cria a janela com as dimensoes definidas
pg.display.set_caption("JOGO DO DILEMA DOS PRISIONEIROS") # define o titulo da janela

# define as tres fontes de texto em tamanhos distintos
arial = pg.font.SysFont("Arial", 30)
arial_pequeno = pg.font.SysFont("Arial", 22)
arial_huge = pg.font.SysFont("Arial", 50)

# =========================================================
# CORES 
# (tuplas RGB)
# =========================================================

branco = (230,230,230)
preto = (0,0,0)
cor_do_fundo = (30,30,30)

verdin = (40,147,47)
vermelhin = (150,3,3)
bege = (194,176,146)
cinza = (50,50,50)
azulzin = (50,150,250)

# =========================================================
# RETANGULOS
# (x, y, largura, altura)
# =========================================================

botao_cooperar = pg.Rect(50, 280, 200, 50)  # botao COOPERAR nas telas de jogo
botao_trair = pg.Rect(50, 340, 200, 50)     # botao TRAIR nas telas de jogo

tabela_escolhas = pg.Rect(30, 200, 570, 220) # painel de fundo que agrupa as escolhas

etapa_nao_lembra = pg.Rect(60, 270, 250, 300)   # caixa do grupo "sem memoria"
bot_random = pg.Rect(83, 350, 200, 50)
bot_bom = pg.Rect(83, 420, 200, 50)
bot_mal = pg.Rect(83, 490, 200, 50)
etapa_lembra = pg.Rect(360, 270, 250, 300)       # caixa do grupo "com memoria"
bot_vingativo = pg.Rect(383, 350, 200, 50)
bot_imitador = pg.Rect(383, 420, 200, 50)
bot_bravo = pg.Rect(383, 490, 200, 50)

pygame-ce 2.5.7 (SDL 2.32.10, Python 3.13.7)


### COMPORTAMENTOS DOS BOTS E LÓGICA DO JOGO

O jogo conta com seis bots, divididos em dois grupos conforme o uso ou não de memória das rodadas anteriores. Os três primeiros - Randômico, Bonzinho e Malvadão - não consideram o histórico do jogo: o Randômico escolhe entre trair e cooperar de forma aleatória a cada rodada, o Bonzinho sempre coopera e o Malvadão sempre trai, independentemente do que o jogador tenha escolhido anteriormente.

Os outros três - Vingativo, Imitador e Bravo - utilizam as variáveis `T` (número total de traições do jogador), `c` (cooperações consecutivas do jogador) e `t` (traições consecutivas do jogador) para adaptar sua estratégia. O Vingativo começa cooperando, mas ao ser traído uma única vez passa a trair em todas as rodadas seguintes, sem possibilidade de reconciliação. O Imitador coopera na primeira rodada e depois copia exatamente o que o jogador fez na rodada anterior. O Bravo é um bot intermediário entre o vingativo e o imitador: responde a uma traição traindo duas vezes seguidas, mas volta a cooperar se o jogador não traí-lo novamente.

A lógica de pontuação foi implementada na função `payoff`, que retorna a variação de pontos do jogador - sempre negativa, pois representa anos de pena. Quanto menor (mais negativo) o total acumulado em `total_points`, pior o desempenho. O objetivo do jogador, explicitado na tela das instruções, é minimizar essa pena total, o que o leva a descobrir qual é a estratégia ótima para cada tipo de adversário.

In [2]:
# =========================================================
# COMPORTAMENTO DOS BOTS
# =========================================================

machine_choices = ["trair", "cooperar"] # opcoes possiveis para o bot

def randomico():
    """Retorna uma escolha aleatória entre trair e cooperar."""
    return choice(machine_choices)

def bonzinho():
    """Sempre coopera, independentemente das decisões anteriores do jogador."""
    return "cooperar"

def malvado():
    """Sempre trai, independentemente das decisões anteriores do jogador."""
    return "trair"

def vingativo():
    """Coopera enquanto o jogador nunca tiver traído (T == 0).
    Assim que ocorre uma traição, passa a trair em todas as rodadas seguintes.
    Usa a variável global T (total de traições do jogador)."""
    global T
    if T == 0:
        return "cooperar"
    else: 
        return "trair"
    
def imitador():
    """Implementa o Tit-for-Tat: coopera na primeira rodada e depois
    copia a última jogada do jogador. Se o jogador cooperou por último
    (t == 0), coopera; se traiu por último (c == 0), trai.
    Usa as variáveis globais c (cooperações consecutivas) e t (traições consecutivas)."""
    global c
    global t
    if c == 0 and t == 0: # primeira rodada
        return "cooperar"
    elif c == 0: # jogador traiu por ultimo
        return "trair"
    elif t == 0: # jogador cooperou por ultimo
        return "cooperar"
     
def bravo():
    """Variante do Imitador: coopera na primeira rodada, mas após uma traição
    responde traindo duas vezes seguidas (só volta a cooperar depois que o jogador 
    cooperar ao menos duas vezes seguidas (c >= 2)). Usa as variáveis globais c (cooperações 
    consecutivas), t (traições consecutivas) e T (traições totais)."""
    global c
    global t
    global T
    if c == 0 and t == 0: # primeira rodada
        return "cooperar"
    elif c < 2 and T > 0: # jogador já traiu e ainda nao reconquistou a confianca
        return "trair"
    else: # jogador cooperou pelo menos duas vezes seguidas
        return "cooperar"

#========================================================
# LOGICA DO JOGO
# =========================================================

def payoff(machine_choice, user_choice):
    """Calcula e retorna a variação de pontos do jogador para uma rodada.
    Os valores são negativos pois representam anos de pena de prisão.
    Tabela de resultados:
        jogador trai  + bot trai      -> -5
        jogador trai  + bot coopera   -> -1
        jogador coop. + bot trai      -> -10
        jogador coop. + bot coopera   -> -2
    """
    payoff = 0
    if user_choice == "trair":
        if machine_choice == "trair":
            payoff -= 5
        else:
            payoff -= 1
    elif user_choice == "cooperar":
        if machine_choice == "trair":
            payoff -= 10
        else:
            payoff -= 2
    return payoff

#### FUNÇÕES DE DESENHO E ANIMAÇÃO

Cada tela do jogo possui uma função de desenho dedicada. Todas seguem a mesma lógica: apagam o quadro anterior com `tela.fill(cor_do_fundo)`, que preenche a tela inteira com a cor de fundo e efetivamente apaga tudo que estava desenhado antes; renderizam superfícies de texto com `fonte.render()`, que converte uma string em um objeto gráfico; e os posicionam na tela com `tela.blit()`, que "cola" uma superfície em uma posição específica.

A função `resetar` é chamada sempre que o jogador troca de fase ou reinicia, restaurando todas as variáveis de controle - contadores de traição, cooperações consecutivas, pontuação total e estado da animação - aos seus valores iniciais.

A função `desenhar_escolhas` é compartilhada por todas as telas de jogo: ela exibe os botões de COOPERAR e TRAIR, o placar acumulado, o contador de traições e cooperações do jogador e - após cada rodada - a escolha do bot e o resultado em anos de cadeia. Os botões mudam de cor em tempo real conforme o cursor passa sobre eles, o que é possível porque `desenhar_escolhas` recebe `mouse_pos` como argumento e verifica a colisão a cada quadro com `.collidepoint()`.

A animação do menu é controlada pelas variáveis `frame_atual`, `tempo_frame` e `animacao_terminou`. A cada chamada de `desenhar_menu`, `tempo_frame` é incrementado; quando ultrapassa `velocidade_animacao`, o programa avança para o próximo sprite da lista `frames_cadeia`. Quando todos os frames foram exibidos, `animacao_terminou` é marcado como `True` e o último frame é mantido fixo.

In [3]:
# =========================================================
# FUNCOES AUXILIARES E DE DESENHO
# =========================================================

def resetar():
    """Restaura todas as variáveis de controle do jogo aos valores iniciais.
    Deve ser chamada ao trocar de fase ou ao pressionar R para reiniciar.
    Afeta: pode_clicar, user_choice, machine_choice, resultado, total_points,
    C, T, c, t, frame_atual, tempo_frame, animacao_terminou.
    """
    global pode_clicar
    pode_clicar = True
    global user_choice
    user_choice = None
    global machine_choice
    machine_choice = None
    global resultado
    resultado = 0
    global total_points
    total_points = 0
    global C
    C = 0
    global T
    T = 0
    global c
    c = 0
    global t
    t = 0
    global frame_atual
    frame_atual = 0
    global tempo_frame
    tempo_frame = 0
    global animacao_terminou
    animacao_terminou = False

    return (
        pode_clicar, user_choice, machine_choice, 
        resultado, total_points, C, T, c, t,
        frame_atual, tempo_frame, animacao_terminou)

# As imagens sao carregadas fora das funcoes para evitar que sejam reecarregadas a cada quadro
frame_1_cadeia = pg.image.load(
    "FINAL_PCD_IMAGENS/image-1.png"
    ).convert_alpha()
frame_2_cadeia = pg.image.load(
    "FINAL_PCD_IMAGENS/image-2.png"
    ).convert_alpha()
frame_3_cadeia = pg.image.load(
    "FINAL_PCD_IMAGENS/image-3.png"
    ).convert_alpha()
frame_4_cadeia = pg.image.load(
    "FINAL_PCD_IMAGENS/image-4.png"
    ).convert_alpha()
frame_5_cadeia = pg.image.load(
    "FINAL_PCD_IMAGENS/image-5.png"
    ).convert_alpha()

# redimensiona todos os frames para o mesmo tamanho com suavizacao - smoothsale
frame_1_cadeia = pg.transform.smoothscale(frame_1_cadeia,(1800,1800))
frame_2_cadeia = pg.transform.smoothscale(frame_2_cadeia,(1800,1800))
frame_3_cadeia = pg.transform.smoothscale(frame_3_cadeia,(1800,1800))
frame_4_cadeia = pg.transform.smoothscale(frame_4_cadeia,(1800,1800))
frame_5_cadeia = pg.transform.smoothscale(frame_5_cadeia,(1800,1800))

frames_cadeia = [frame_1_cadeia, frame_2_cadeia, frame_3_cadeia, frame_4_cadeia, frame_5_cadeia]

# posicao de exibicao dos frames 
# (desloquei levemente para mostrar so a parte relevante da imagem)
onde_frames_cadeia = ((155-largura), (-7.5-altura))

# variaveis de controle da animacao do menu
frame_atual = 0          # indice do frame sendo exibido
tempo_frame = 0          # contador de ciclos desde o ultimo troca de frame
velocidade_animacao = 180 # numero de ciclos por frame
animacao_terminou = False

def desenhar_menu():
    """Desenha a tela de menu com os controles do jogo e a animação de abertura da cela.
    A animação percorre a lista frames_cadeia quadro a quadro, avançando um frame
    a cada velocidade_animacao ciclos do loop principal.
    """
    tela.fill(cor_do_fundo)
    texto_menu_titulo = arial_huge.render("MENU", True, branco)
    texto_menu_1 = arial.render("> : AVANÇAR", True, branco)
    texto_menu_2 = arial.render("< : VOLTAR", True, branco)
    texto_menu_3 = arial.render("ESPAÇO : FECHAR", True, branco)

    tela.blit(texto_menu_titulo, (290,100))
    tela.blit(texto_menu_1, (240,160))
    tela.blit(texto_menu_2, (240,200))
    tela.blit(texto_menu_3, (240,240))

    global tempo_frame
    global frame_atual
    global animacao_terminou

    tela.blit(frames_cadeia[frame_atual], onde_frames_cadeia) # exibe o frame atual

    if not animacao_terminou:
        tempo_frame += 1
        if tempo_frame >= velocidade_animacao:   # hora de trocar de frame
            tempo_frame = 0
            if frame_atual < len(frames_cadeia)-1:
                frame_atual += 1
            else:
                animacao_terminou = True          # todos os frames foram exibidos

# carrega e redimensiona a imagem do jornal
jornal_intro = pg.image.load(
    "FINAL_PCD_IMAGENS/Captura de tela 2026-05-11 105120.png"
).convert_alpha()
jornal_intro = pg.transform.smoothscale(jornal_intro, (546,585))

# calcula a posicao para centralizar o jornal na janela
onde_jornal_intro = ((largura-546)/2, (altura-585)/2)

def desenhar_jornal():
    """Desenha a tela do jornal fictício de contextualização, centralizado na janela."""
    tela.fill(cor_do_fundo)
    tela.blit(jornal_intro, onde_jornal_intro)


def desenhar_tela_instrucoes():
    """Desenha a tela de instruções com o texto explicativo e a tabela de payoffs.
    A tabela é gerada por uma função interna (desenhar_tabela),que usa laços para 
    desenhar as linhas e centralizar os textos em cada célula.
    """
    tela.fill(cor_do_fundo)

    texto_tela_instrucoes_1 = arial_huge.render("INSTRUÇÕES:", True, branco)
    texto_tela_instrucoes_2 = arial_pequeno.render(
        "Você, como um prisioneiro sob custódia, tem um objetivo: tentar achar", True, branco)
    texto_tela_instrucoes_3 = arial_pequeno.render(
        "qual é a estratégia ótima - TRAIR ou COOPERAR - para cada situação", True, branco)
    texto_tela_instrucoes_4 = arial_pequeno.render(
        "(lembre-se de que estratégia ótima é aquela que resultará na pena total", True, branco)
    texto_tela_instrucoes_5 = arial_pequeno.render(
        "mínima). Aqui está a tabela explicando as penas:", True, branco)

    tela.blit(texto_tela_instrucoes_1, (50,70))
    tela.blit(texto_tela_instrucoes_2, (50,130))
    tela.blit(texto_tela_instrucoes_3, (50,160))
    tela.blit(texto_tela_instrucoes_4, (50,190))
    tela.blit(texto_tela_instrucoes_5, (50,220))
    
    def desenhar_tabela():
        """Desenha a grade da tabela de payoffs e preenche cada célula com o texto correto.
        As linhas da grade sao desenhadas com pg.draw.line e os textos sao centralizados
        em cada celula usando get_rect(center=...).
        """
        tabela_instrucoes = pg.Rect(40, 280, 600, 240)
        pg.draw.rect(tela, cinza, tabela_instrucoes)

        x = 40
        y = 280
        largura_tabela = 200
        altura_tabela = 80

        for i in range(4): # linhas horizontais
            pg.draw.line(tela, bege,
                (x, y + i*altura_tabela),
                (x + 3*largura_tabela, y + i*altura_tabela), 1)

        for i in range(4): # linhas verticais
            pg.draw.line(tela, bege,
                (x + i*largura_tabela, y),
                (x + i*largura_tabela, y + 3*altura_tabela), 1)

        textos = [
            [" PLAYER 1 | PLAYER 2  ", "COOPERAR", "TRAIR"],
            ["COOPERAR", "2 anos | 2 anos", "10 anos | 1 ano"],
            ["TRAIR", "1 ano | 10 anos", "5 anos | 5 anos"]
        ]

        for linha in range(3):
            for coluna in range(3):
                texto = arial_pequeno.render(textos[linha][coluna], True, branco)
                centro_x = x + coluna*largura_tabela + largura_tabela/2
                centro_y = y + linha*altura_tabela + altura_tabela/2
                pos = texto.get_rect(center=(centro_x, centro_y))
                tela.blit(texto, pos)
        
    desenhar_tabela()


def desenhar_tela_fases():
    """Desenha a tela de seleção de fases com os dois grupos de bots.
    Os botões de cada bot sao clicáveis: ao clicar, o loop principal detecta
    a colisão (ele capta a posição do mouse e se houve um clique enquanto
    o mouse estava em cima do botão) e redireciona para a fase correspondente.
    Os nomes dos bots sao centralizados em seus respectivos botões usando get_rect(center=...).
    """
    tela.fill(cor_do_fundo)

    texto_tela_fases_1 = arial_huge.render("SITUACOES:", True, branco)
    texto_tela_fases_2 = arial_pequeno.render(
        "Em cada uma das situações, você vai encontrar um 'bot' diferente, ca-", True, branco)
    texto_tela_fases_3 = arial_pequeno.render(
        "da qual com um comportamento distinto (exemplo: um sempre trai, outro", True, branco)
    texto_tela_fases_4 = arial_pequeno.render(
        "escolhe aleatoriamente, etc). Aqui embaixo, você verá que as fases estão", True, branco)
    texto_tela_fases_5 = arial_pequeno.render(
        "divididas em dois grupos. Você entenderá isso depois...", True, branco)
    
    tela.blit(texto_tela_fases_1, (50,70))
    tela.blit(texto_tela_fases_2, (50,130))
    tela.blit(texto_tela_fases_3, (50,160))
    tela.blit(texto_tela_fases_4, (50,190))
    tela.blit(texto_tela_fases_5, (50,220))

    texto_etapa_nao_lembra = arial.render("SEM MEMÓRIA:", True, branco)
    texto_etapa_lembra = arial.render("COM MEMÓRIA:", True, branco)

    # desenha os paineis de grupo e os botoes individuais
    pg.draw.rect(tela, cinza, etapa_nao_lembra)
    tela.blit(texto_etapa_nao_lembra, (86, 290))
    pg.draw.rect(tela, bege, bot_random)
    pg.draw.rect(tela, bege, bot_bom)
    pg.draw.rect(tela, bege, bot_mal)

    pg.draw.rect(tela, cinza, etapa_lembra)
    tela.blit(texto_etapa_lembra, (386, 290))
    pg.draw.rect(tela, bege, bot_vingativo)
    pg.draw.rect(tela, bege, bot_imitador)
    pg.draw.rect(tela, bege, bot_bravo)

    nomes_bots = [
        ("RANDOM", bot_random), ("BONZINHO", bot_bom), ("MALVADÃO", bot_mal),
        ("VINGATIVO", bot_vingativo), ("IMITADOR", bot_imitador), ("BRAVO", bot_bravo)
    ]

    for nome, botao in nomes_bots: # centraliza o nome dentro de cada botao
        texto = arial_pequeno.render(nome, True, preto)
        pos_texto = texto.get_rect(center=botao.center)
        tela.blit(texto, pos_texto)


def desenhar_escolhas(
    machine_choice, user_choice, mouse_pos, pode_clicar,
    botao_cooperar, botao_trair, total_points, T, C
):
    """Desenha o painel de jogo compartilhado por todas as fases.
    Exibe os botoes COOPERAR e TRAIR (com mudança de cor ao passar o mouse),
    a última escolha do bot, o resultado da rodada em anos de cadeia,
    a pena total acumulada e os contadores de traições e cooperações do jogador.
    O resultado so aparece depois que tanto o jogador quanto o bot jogaram
    ("if machine_choice and user_choice").
    """
    pg.draw.rect(tela, cinza, tabela_escolhas)

    texto_user_choice = arial.render("VOCÊ ESCOLHE:", True, branco)
    texto_cooperar = arial.render("COOPERAR", True, preto)
    onde_texto_cooperar = texto_cooperar.get_rect(center=botao_cooperar.center)
    texto_trair = arial.render("TRAIR", True, preto)
    onde_texto_trair = texto_trair.get_rect(center=botao_trair.center)

    tela.blit(texto_user_choice, (50,220))

    # botao COOPERAR muda para verde quando o cursor esta sobre ele
    cor_cooperar = bege
    if botao_cooperar.collidepoint(mouse_pos):
        cor_cooperar = verdin
    pg.draw.rect(tela, cor_cooperar, botao_cooperar)
    tela.blit(texto_cooperar, onde_texto_cooperar)

    # botao TRAIR muda para vermelho quando o cursor esta sobre ele
    cor_trair = bege
    if botao_trair.collidepoint(mouse_pos):
        cor_trair = vermelhin
    pg.draw.rect(tela, cor_trair, botao_trair)
    tela.blit(texto_trair, onde_texto_trair)

    texto_machine_choice = arial.render("BOT ESCOLHE:", True, branco)
    tela.blit(texto_machine_choice, (340,220))

    if machine_choice and user_choice: # so exibe a escolha do bot apos a primeira jogada
        texto_bot = arial.render(machine_choice.upper(), True, branco)
        tela.blit(texto_bot, (340,290))

    if resultado == 0: # antes da primeira jogada exibe tracejado
        texto_resultado = arial.render(f'Anos de cadeia: ____', True, branco)
    else:
        texto_resultado = arial.render(f"Anos de cadeia: {-resultado}", True, branco)
    tela.blit(texto_resultado, (340,350))
    
    texto_reset = arial_pequeno.render("PRESSIONE R PARA REINICIAR", True, branco)
    tela.blit(texto_reset, (50,20))

    texto_pontos_totais = arial.render(f'PENA TOTAL: {-total_points}', True, branco)
    tela.blit(texto_pontos_totais,(410,520))

    texto_trair_total = arial.render(f'Você TRAIU: {T}', True, branco)
    texto_cooperar_total = arial.render(f'Você COOPEROU: {C}', True, branco)
    tela.blit(texto_cooperar_total, (50,500))
    tela.blit(texto_trair_total, (50,540))


def desenhar_tela_random(
    machine_choice, user_choice, mouse_pos, pode_clicar,
    botao_cooperar, botao_trair, total_points, T, C
):
    """Desenha a tela de jogo contra o Bot Randômico.
    Exibe o nome e a descrição do bot e usa da função desenhar_escolhas
    pré-definida para desenhar o painel de escolhas.
    """
    texto_tela_random_1 = arial_huge.render("BOT RANDOM", True, branco)
    texto_tela_random_2 = arial_pequeno.render("Ele joga aleatoriamente", True, branco)
    tela.fill(cor_do_fundo)
    tela.blit(texto_tela_random_1, (50,60))
    tela.blit(texto_tela_random_2, (50,110))
    desenhar_escolhas(machine_choice, user_choice, mouse_pos, pode_clicar,
        botao_cooperar, botao_trair, total_points, T, C)


def desenhar_tela_bonzinho(
    machine_choice, user_choice, mouse_pos, pode_clicar,
    botao_cooperar, botao_trair, total_points, T, C
):
    """Desenha a tela de jogo contra o Bot Bonzinho."""
    tela.fill(cor_do_fundo)
    desenhar_escolhas(machine_choice, user_choice, mouse_pos, pode_clicar,
        botao_cooperar, botao_trair, total_points, T, C)
    texto_tela_bonzinho_1 = arial_huge.render("BOT BONZINHO", True, branco)
    texto_tela_bonzinho_2 = arial_pequeno.render("Ele sempre irá cooperar", True, branco)
    tela.blit(texto_tela_bonzinho_1, (50,60))
    tela.blit(texto_tela_bonzinho_2, (50,110))


def desenhar_tela_malvado(
    machine_choice, user_choice, mouse_pos, pode_clicar,
    botao_cooperar, botao_trair, total_points, T, C
):
    """Desenha a tela de jogo contra o Bot Malvadão."""
    tela.fill(cor_do_fundo)
    desenhar_escolhas(machine_choice, user_choice, mouse_pos, pode_clicar,
        botao_cooperar, botao_trair, total_points, T, C)
    texto_tela_malvado_1 = arial_huge.render("BOT MALVADÃO", True, branco)
    texto_tela_malvado_2 = arial_pequeno.render("Ele sempre irá trair", True, branco)
    tela.blit(texto_tela_malvado_1, (50,60))
    tela.blit(texto_tela_malvado_2, (50,110))


def desenhar_tela_vingativo(
    machine_choice, user_choice, mouse_pos, pode_clicar,
    botao_cooperar, botao_trair, total_points, T, C
):
    """Desenha a tela de jogo contra o Bot Vingativo."""
    tela.fill(cor_do_fundo)
    desenhar_escolhas(machine_choice, user_choice, mouse_pos, pode_clicar,
        botao_cooperar, botao_trair, total_points, T, C)
    texto_tela_vingativo_1 = arial_huge.render("BOT VINGATIVO", True, branco)
    texto_tela_vingativo_2 = arial_pequeno.render(
        "Ele é bonzinho, mas se você trair ele uma só vez, ele fica malvadão", True, branco)
    tela.blit(texto_tela_vingativo_1, (50,60))
    tela.blit(texto_tela_vingativo_2, (50,110))


def desenhar_tela_imitador(
    machine_choice, user_choice, mouse_pos, pode_clicar,
    botao_cooperar, botao_trair, total_points, T, C
):
    """Desenha a tela de jogo contra o Bot Imitador."""
    tela.fill(cor_do_fundo)
    desenhar_escolhas(machine_choice, user_choice, mouse_pos, pode_clicar,
        botao_cooperar, botao_trair, total_points, T, C)
    texto_tela_imitador_1 = arial_huge.render("BOT IMITADOR", True, branco)
    texto_tela_imitador_2 = arial_pequeno.render(
        "Ele imita o que você jogou na última rodada", True, branco)
    tela.blit(texto_tela_imitador_1, (50,60))
    tela.blit(texto_tela_imitador_2, (50,110))


def desenhar_tela_bravo(
    machine_choice, user_choice, mouse_pos, pode_clicar,
    botao_cooperar, botao_trair, total_points, T, C
):
    """Desenha a tela de jogo contra o Bot Bravo."""
    tela.fill(cor_do_fundo)
    desenhar_escolhas(machine_choice, user_choice, mouse_pos, pode_clicar,
        botao_cooperar, botao_trair, total_points, T, C)
    texto_tela_bravo_1 = arial_huge.render("BOT BRAVO", True, branco)
    texto_tela_bravo_2 = arial_pequeno.render(
        "Se você trair ele, ele retalia traindo duas vezes seguidas.", True, branco)
    tela.blit(texto_tela_bravo_1, (50,60))
    tela.blit(texto_tela_bravo_2, (50,110))


def desenhar_tela_explicando():
    """Desenha a tela de explicação teórica exibida após todas as fases,
    explicando o conceito de memória dos bots e as estratégias ótimas
    para cada grupo.
    """
    tela.fill(cor_do_fundo)
    texto_tela_explicando_1 = arial_huge.render("MEMÓRIA", True, branco)
    texto_tela_explicando_2 = arial_pequeno.render(
        "Se você estava se perguntando o que significa a 'memória' dos", True, branco)
    texto_tela_explicando_3 = arial_pequeno.render(
        "bots, essa 'memória' consiste na capacidade de escolher trair ou", True, branco)
    texto_tela_explicando_4 = arial_pequeno.render(
        "cooperar com base no que você jogou antes (é como se ele se", True, branco)
    texto_tela_explicando_5 = arial_pequeno.render(
        "lembrasse das ações anteriores). ", True, branco)
    texto_tela_explicando_6 = arial_huge.render("ESTRATÉGIAS ÓTIMAS", True, branco)
    texto_tela_explicando_7 = arial_pequeno.render(
        "Para os bots sem memória, o ideal é sempre traí-los, pois é a ação", True, branco)
    texto_tela_explicando_8 = arial_pequeno.render(
        "que lhe dará o maior benefício e o bot terá o mesmo comportamento", True, branco)
    texto_tela_explicando_9 = arial_pequeno.render(
        "independentemente do que você jogar. MAS, quando eles tem memória,", True, branco)
    texto_tela_explicando_10 = arial_pequeno.render(
        "irão 'lembrar' que você traiu e vão agir de acordo, então a estratégia", True, branco)
    texto_tela_explicando_11 = arial_pequeno.render(
        "ótima passa ser a cooperação!", True, branco)
    tela.blit(texto_tela_explicando_1, (50,60))
    tela.blit(texto_tela_explicando_2, (50,120))
    tela.blit(texto_tela_explicando_3, (50,150))
    tela.blit(texto_tela_explicando_4, (50,180))
    tela.blit(texto_tela_explicando_5, (50,210))
    tela.blit(texto_tela_explicando_6, (50,320))
    tela.blit(texto_tela_explicando_7, (50,380))
    tela.blit(texto_tela_explicando_8, (50,410))
    tela.blit(texto_tela_explicando_9, (50,440))
    tela.blit(texto_tela_explicando_10, (50,470))
    tela.blit(texto_tela_explicando_11, (50,500))


def desenhar_tela_fim():
    """Desenha a tela final com a mensagem de encerramento e as referências do projeto."""
    tela.fill(cor_do_fundo)
    texto_tela_fim_1 = arial_huge.render("FIM", True, branco)
    texto_tela_fim_2 = arial_pequeno.render(
        "Parabéns!! Prisioneiro, prisioneiro, você foi inocentado!", True, branco)
    texto_tela_fim_3 = arial_pequeno.render(
        "Parece que tem um clone maligno seu andando por aí... Bom, pelo me-", True, branco)
    texto_tela_fim_4 = arial_pequeno.render(
        "nos você aprendeu um pouco mais sobre teoria dos jogos e pôde", True, branco)
    texto_tela_fim_5 = arial_pequeno.render(
        "experenciar o dilema dos prisioneiros. Espero que tenha gostado!", True, branco)
    texto_tela_fim_6 = arial_huge.render("REFERÊNCIAS", True, branco)
    texto_tela_fim_7 = arial_pequeno.render("SITE DO TRUST LA", True, branco)
    texto_tela_fim_8 = arial_pequeno.render(
        "VIDEO VERITASSIUM DILEMA DOS PRISIONEIROS", True, branco)
    texto_tela_fim_9 = arial_pequeno.render("livro introduction to game theory", True, branco)
    texto_tela_fim_10 = arial_pequeno.render("PYTHON GERALZAO", True, branco)
    texto_tela_fim_11 = arial_pequeno.render("PYGAME", True, branco)
    tela.blit(texto_tela_fim_1, (50,60))
    tela.blit(texto_tela_fim_2, (50,120))
    tela.blit(texto_tela_fim_3, (50,150))
    tela.blit(texto_tela_fim_4, (50,180))
    tela.blit(texto_tela_fim_5, (50,210))
    tela.blit(texto_tela_fim_6, (50,320))
    tela.blit(texto_tela_fim_7, (50,380))
    tela.blit(texto_tela_fim_8, (50,410))
    tela.blit(texto_tela_fim_9, (50,440))
    tela.blit(texto_tela_fim_10, (50,470))
    tela.blit(texto_tela_fim_11, (50,500))

### LOOP PRINCIPAL E MÁQUINA DE ESTADOS

O coração do programa é um loop `while True` (por meio da variável booleana `rodando`) que roda continuamente até que o usuário feche o jogo. A cada iteração, o loop captura os eventos do Pygame com `pg.event.get()` - que retorna uma lista de tudo que aconteceu desde o último quadro (teclas, cliques, fechamento da janela) -, atualiza as variáveis de estado com base nesses eventos, chama a função de desenho correspondente à tela atual e usa `pg.display.flip()` para enviar o quadro recém-desenhado para a tela.

A navegação entre as diferentes telas é gerenciada pela variável `estado`, que percorre a sequência `"menu"`, `"jornal"`, `"tela_instrucoes"`, `"tela_fases"` e, a partir daí, qualquer uma das seis telas de jogo, terminando em `"tela_explicando"` e `"tela_fim"`. Utilizar de uma variável para determinar qual é a tela a ser desenhada e os respectivos eventos a serem processados consiste numa técnica chamada de máquina de estados.

Após cada jogada, o programa chama `pg.display.flip()` imediatamente para mostrar o resultado na tela e em seguida executa `pg.time.wait(1000)`, que pausa o loop por 1 segundo antes de liberar o próximo clique. Durante esse intervalo, `pode_clicar` é mantido como `False`, impedindo que um clique acidental seja registrado enquanto o resultado ainda está sendo exibido. Só após a espera `total_points` é atualizado com o resultado da rodada e `pode_clicar` volta a ser `True`. Na tela de fases, o jogador também pode clicar diretamente nos botões dos bots para saltar a qualquer fase, o que chama `resetar()` e atualiza `estado` antes de continuar o loop.

In [4]:
# =========================================================
# VARIAVEIS DO JOGO
# =========================================================

estado = "menu"        # controla qual tela esta sendo exibida
rodando = True         # controla o loop principal - varia somente entre True ou False
pode_clicar = True     # bloqueia cliques durante a pausa apos cada jogada
user_choice = None     # ultima escolha do jogador
machine_choice = None  # ultima escolha do bot
resultado = 0          # variacao de pontos da ultima rodada
total_points = 0       # acumulado de pontos (negativo = anos de pena)
C = 0  # total de cooperacoes do jogador na fase atual
T = 0  # total de traicoes do jogador na fase atual
c = 0  # cooperacoes consecutivas (usadas pelos bots com memoria)
t = 0  # traicoes consecutivas (usadas pelos bots com memoria)

# =========================================================
# LOOP PRINCIPAL
# =========================================================

while rodando:

    mouse_pos = pg.mouse.get_pos() # posicao atual do cursor (atualizada a cada quadro)

    for event in pg.event.get(): # percorre todos os eventos desde o ultimo quadro

        if event.type == pg.QUIT: # usuario fechou a janela
            pg.quit()
            sys.exit()
    
        elif event.type == pg.KEYDOWN: # uma tecla foi pressionada
            resetar()
            if event.key == pg.K_SPACE:
                rodando = False
            elif event.key == pg.K_r:
                resetar()
                
            # cada bloco abaixo so e executado se o estado correspondente estiver ativo
            if estado == "menu":
                if event.key == pg.K_RIGHT:
                    estado = "jornal"

            elif estado == "jornal":
                if event.key == pg.K_RIGHT:
                    estado = "tela_instrucoes"
                elif event.key == pg.K_LEFT:
                    estado = "menu"

            elif estado == "tela_instrucoes":
                if event.key == pg.K_LEFT:
                    estado = "jornal"
                elif event.key == pg.K_RIGHT:
                    estado = "tela_fases"

            elif estado == "tela_fases":
                if event.key == pg.K_LEFT:
                    estado = "tela_instrucoes"
                elif event.key == pg.K_RIGHT:
                    estado = "tela_random"

            elif estado == "tela_random":
                if event.key == pg.K_LEFT:
                    estado = "tela_instrucoes"
                elif event.key == pg.K_RIGHT:
                    estado = "tela_bonzinho"

            elif estado == "tela_bonzinho":
                if event.key == pg.K_LEFT:
                    estado = "tela_random"
                elif event.key == pg.K_RIGHT:
                    estado = "tela_malvado"

            elif estado == "tela_malvado":
                if event.key == pg.K_LEFT:
                    estado = "tela_bonzinho"
                if event.key == pg.K_RIGHT:
                    estado = "tela_vingativo"

            elif estado == "tela_vingativo":
                if event.key == pg.K_LEFT:
                    estado = "tela_bonzinho"
                if event.key == pg.K_RIGHT:
                    estado = "tela_imitador"
            
            elif estado == "tela_imitador":
                if event.key == pg.K_LEFT:
                    estado = "tela_vingativo"
                if event.key == pg.K_RIGHT:
                    estado = "tela_bravo"

            elif estado == "tela_bravo":
                if event.key == pg.K_LEFT:
                    estado = "tela_imitador"
                if event.key == pg.K_RIGHT:
                    estado = "tela_explicando"
            
            elif estado == "tela_explicando":
                if event.key == pg.K_LEFT:
                    estado = "tela_bravo"
                if event.key == pg.K_RIGHT:
                    estado = "tela_fim"

            elif estado == "tela_fim":
                if event.key == pg.K_LEFT:
                    estado = "tela_explicando"
                if event.key == pg.K_RIGHT:
                    estado = "menu"


        if event.type == pg.MOUSEBUTTONDOWN: # botao do mouse foi pressionado
                if event.button == 1 and pode_clicar: # so processa clique esquerdo e se liberado

                    if estado == "tela_fases": # cliques nos botoes de selecao de fase
                        if bot_random.collidepoint(mouse_pos):
                            resetar()
                            estado = "tela_random"
                            continue
                        if bot_bom.collidepoint(mouse_pos):
                            resetar()
                            estado = "tela_bonzinho"
                            continue
                        if bot_mal.collidepoint(mouse_pos):
                            resetar()
                            estado = "tela_malvado"
                            continue
                        if bot_vingativo.collidepoint(mouse_pos):
                            resetar()
                            estado = "tela_vingativo"
                            continue
                        if bot_imitador.collidepoint(mouse_pos):
                            resetar()
                            estado = "tela_imitador"
                            continue

                    # para cada fase: verifica qual botao foi clicado, registra a escolha
                    # do jogador, consulta o bot correspondente, calcula o payoff
                    # e atualiza os contadores de memoria (C, T, c, t)

                    if estado == "tela_random":
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = randomico()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1; t = 0; c += 1
                            pode_clicar = False
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = randomico()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1; t += 1; c = 0
                            pode_clicar = False
            
                    if estado == "tela_bonzinho":
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = bonzinho()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1; t = 0; c += 1
                            pode_clicar = False
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = bonzinho()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1; t += 1; c = 0
                            pode_clicar = False

                    if estado == "tela_malvado":
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = malvado()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1; t = 0; c += 1
                            pode_clicar = False
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = malvado()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1; t += 1; c = 0
                            pode_clicar = False

                    if estado == "tela_vingativo":
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = vingativo()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1; t = 0; c += 1
                            pode_clicar = False
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = vingativo()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1; t += 1; c = 0
                            pode_clicar = False

                    if estado == "tela_imitador":
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = imitador()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1; t = 0; c += 1
                            pode_clicar = False
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = imitador()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1; t += 1; c = 0
                            pode_clicar = False

                    if estado == "tela_bravo":
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = bravo()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1; t = 0; c += 1
                            pode_clicar = False
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = bravo()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1; t += 1; c = 0
                            pode_clicar = False


    # =====================================================
    # DESENHO DAS TELAS
    # =====================================================

    if estado == "menu":
        desenhar_menu()
    elif estado == "jornal":
        desenhar_jornal()
    elif estado == "tela_instrucoes":
        desenhar_tela_instrucoes()
    elif estado == "tela_fases":
        desenhar_tela_fases()
    elif estado == "tela_random":
        desenhar_tela_random(
            machine_choice, user_choice, mouse_pos, pode_clicar,
            botao_cooperar, botao_trair, total_points, T, C)
        if not pode_clicar: # apos uma jogada: mostra resultado, espera 1s e libera clique
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True
    elif estado == "tela_bonzinho":
        desenhar_tela_bonzinho(
            machine_choice, user_choice, mouse_pos, pode_clicar,
            botao_cooperar, botao_trair, total_points, T, C)
        if not pode_clicar:
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True
    elif estado == "tela_malvado":
        desenhar_tela_malvado(
            machine_choice, user_choice, mouse_pos, pode_clicar,
            botao_cooperar, botao_trair, total_points, T, C)
        if not pode_clicar:
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True
    elif estado == "tela_vingativo":
        desenhar_tela_vingativo(
            machine_choice, user_choice, mouse_pos, pode_clicar,
            botao_cooperar, botao_trair, total_points, T, C)
        if not pode_clicar:
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True
    elif estado == "tela_imitador":
        desenhar_tela_imitador(
            machine_choice, user_choice, mouse_pos, pode_clicar,
            botao_cooperar, botao_trair, total_points, T, C)
        if not pode_clicar:
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True
    elif estado == "tela_bravo":
        desenhar_tela_bravo(
            machine_choice, user_choice, mouse_pos, pode_clicar,
            botao_cooperar, botao_trair, total_points, T, C)
        if not pode_clicar:
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True
    elif estado == "tela_explicando":
        desenhar_tela_explicando()
    elif estado == "tela_fim":
        desenhar_tela_fim()

    pg.display.flip() # envia o quadro desenhado para a tela

pg.quit()
sys.exit()

SystemExit: 

c:\Program Files\Python313\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## RESULTADOS E DISCUSSÃO

Uma vez que o programa estava pronto, para obter opiniões e observar como os jogadores interagiam/aprendiam com o jogo, utilizei de amigos e familiares como usuários teste. O percorrer do jogo mostrou-se intuitivo, visto que o usuário percorre as fases, enfrenta cada um dos seis bots e pode observar como diferentes estratégias se comportam ao longo das rodadas. No entanto, a reação dos usuários mostrou-se bastante variada: enquanto alguns ficaram altamente intrigados e buscaram avidamente descobrir e ponderar acerca da estratégia ótima, outros simplesmente passaram para obter as respostas (e mostraram mais interesse na contextualização fictícia do problema). 

Do ponto de vista da Teoria dos Jogos, o jogo consegue ilustrar bem o paradoxo central do Dilema dos Prisioneiros. Contra o Bot Malvadão, o jogador rapidamente percebe que a única forma de evitar a pena máxima de 10 anos por rodada é trair também, pois mesmo que isso resulte em 5 anos, é melhor que 10. Essa é exatamente a lógica do Equilíbrio de Nash: trair é a estratégia dominante num jogo de rodada única, pois nenhum dos jogadores se beneficia por uma mudança unilateral. Contra o Bot Bonzinho, a situação é diferente: cooperar rende apenas 2 anos por rodada, mas trair rende apenas 1 ano, e a decisão de "explorar" o adversário cooperativo é mostra-se como a estratégia ótima. Isso ilustra o problema da cooperação unilateral: ela só é estável se ambos os lados a adotarem.

Os bots com memória introduzem a dimensão mais rica do jogo. O Bot Imitador (Tit-for-Tat) recompensa a cooperação e pune a traição de forma proporcional, tornando a cooperação mútua a estratégia mais vantajosa a longo prazo. O Bot Vingativo, por sua vez, não perdoa: uma traição inicial condena o jogador a penas altas pelo resto da partida, mesmo que ele tente cooperar depois.

Vale mencionar que o projeto, em sua concepção inicial, não envolvia interface gráfica: a ideia original - registrada no primeiro rascunho - não envolvia uma interface gráfica, apenas entradas do usuário em inputs e saídas em prints. A decisão de migrar para o Pygame surgiu ao longo do desenvolvimento, motivada pelo desejo de tornar o jogo mais imersivo e visualmente interessante, e também pelo interesse em explorar uma biblioteca nova, o que representou um desafio técnico adicional considerável. Essa mudança foi responsável pela maior parte do esforço do projeto - mas mostrou-se realmente gratificante!

Do ponto de vista técnico, o maior desafio foi gerenciar o estado do jogo de forma consistente entre as diferentes telas. O uso de variáveis globais (`T`, `C`, `c`, `t`, `total_points`) resolveu o problema funcionalmente, embora uma solução mais robusta talvez usasse um objeto ou dicionário centralizado para encapsular o estado, permitindo uma modularização do projeto. Outro ponto de aprendizado foi a animação por sprites: a lógica de controlar `frame_atual` e `tempo_frame` dentro do loop principal foi uma introdução prática ao conceito de animação quadro a quadro.

## CONCLUSÃO

O projeto cumpriu seus objetivos. Do lado da Teoria dos Jogos, o jogo permite que o usuário vivencie na prática os conceitos de estratégia dominante, Equilíbrio de Nash e cooperação iterada - conceitos que, quando apenas lidos, costumam parecer abstratos. Ao interagir com os bots repetidamente e ver a pena acumulada crescer ou diminuir conforme suas escolhas, o jogador desenvolve uma intuição sobre quando trair é racional e quando cooperar é mais vantajoso.

Do lado da programação, o projeto representou uma síntese dos conceitos aprendidos ao longo do semestre: funções, estruturas condicionais, listas, variáveis de controle e uso de bibliotecas externas. A construção de uma interface gráfica interativa foi o desafio mais significativo - e também o mais recompensador.

Como possíveis melhorias futuras, seria interessante adicionar gráficos ao final de cada fase mostrando a evolução das escolhas ao longo das rodadas (o que aprofundaria a conexão com análise de dados), além de implementar um modo de simulação automática - em que dois bots jogam entre si - para comparar visualmente o desempenho de diferentes estratégias. Isso tornaria o projeto ainda mais próximo dos experimentos clássicos de Axelrod sobre o Dilema dos Prisioneiros Iterado.